In [23]:
pip install qdrant-client sentence-transformers pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from pathlib import Path
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Filter,
    FieldCondition,
    MatchValue,
)

In [25]:
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

CHUNKS_PATH = PROCESSED_DIR / "knowledge_base_chunks.parquet"

COLLECTION_NAME = "agent_arena_knowledge_base"

print("BASE_DIR:", BASE_DIR)
print("CHUNKS_PATH:", CHUNKS_PATH)
print("Exists:", CHUNKS_PATH.exists())

BASE_DIR: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor
CHUNKS_PATH: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\processed\knowledge_base_chunks.parquet
Exists: True


In [26]:
df_chunks = pd.read_parquet(CHUNKS_PATH)

print("Chunks:", len(df_chunks))

df_chunks[
    [
        "context_id",
        "provider",
        "document_type",
        "source_file",
        "section_path",
    ]
].head(10)

Chunks: 298


,context_id,provider,document_type,source_file,section_path
0,CTX-0001,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage
1,CTX-0002,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Purpose
2,CTX-0003,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > When to use t...
3,CTX-0004,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Typical usage...
4,CTX-0005,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Role in docum...
5,CTX-0006,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Example organ...
6,CTX-0007,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Pros
7,CTX-0008,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Cons
8,CTX-0009,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Best suited for
9,CTX-0010,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Not ideal for


In [27]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

embedding_dim = embedding_model.get_sentence_embedding_dimension()

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedding_dim)

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


In [28]:
texts_to_embed = df_chunks["contextualized_chunk_text"].tolist()

embeddings = embedding_model.encode(
    texts_to_embed,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(type(embeddings))
print(embeddings.shape)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches: 100%|██████████| 10/10 [00:13<00:00,  1.40s/it]

<class 'numpy.ndarray'>
(298, 384)


In [29]:
client = QdrantClient(host="localhost", port=6333)

client.get_collections()

C:\Users\Usuario\AppData\Local\Temp\ipykernel_11484\2617498832.py:1: UserWarning: Qdrant client version 1.16.2 is incompatible with server version 1.18.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(host="localhost", port=6333)


CollectionsResponse(collections=[CollectionDescription(name='agent_arena_knowledge_base')])

In [30]:
COLLECTION_NAME = "agent_arena_knowledge_base"

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=embedding_dim,
        distance=Distance.COSINE
    )
)

print(f"Collection '{COLLECTION_NAME}' created.")

Collection 'agent_arena_knowledge_base' created.


In [31]:
payload_columns = [
    "context_id",
    "chunk_id",
    "source_file",
    "source_path",
    "document_title",
    "provider",
    "document_type",
    "section_title",
    "section_path",
    "section_level",
    "section_index",
    "chunk_index",
    "chunk_text",
    "contextualized_chunk_text",
]

points = []

for idx, row in df_chunks.reset_index(drop=True).iterrows():
    payload = {
        col: row[col]
        for col in payload_columns
        if col in row and pd.notna(row[col])
    }

    point = PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload=payload
    )

    points.append(point)

print("Points prepared:", len(points))

Points prepared: 298


In [32]:
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print("Inserted points:", len(points))

Inserted points: 298


In [33]:
collection_info = client.get_collection(COLLECTION_NAME)
collection_info

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=298, segments_count=6, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segmen

In [34]:
def build_provider_filter(provider: str | None = None):
    """
    Build Qdrant filter by provider.
    
    provider can be:
    - None
    - "azure"
    - "aws"
    - "neutral"
    """
    if provider is None:
        return None

    return Filter(
        must=[
            FieldCondition(
                key="provider",
                match=MatchValue(value=provider)
            )
        ]
    )


def retrieve_contexts(
    query: str,
    provider: str | None = None,
    top_k: int = 5
) -> list[dict]:
    """
    Retrieve relevant contexts from Qdrant using the current query_points API.
    """
    query_vector = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).tolist()

    search_filter = build_provider_filter(provider)

    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=search_filter,
        limit=top_k,
        with_payload=True,
        with_vectors=False,
    )

    contexts = []

    for point in response.points:
        payload = dict(point.payload)
        payload["score"] = point.score
        contexts.append(payload)

    return contexts

In [35]:
query = """
I need a document assistant architecture with ingestion, semantic retrieval,
vector search, metadata filtering and agents.
"""

contexts = retrieve_contexts(query, provider=None, top_k=5)

for ctx in contexts:
    print("=" * 120)
    print("Score:", ctx["score"])
    print("Context ID:", ctx["context_id"])
    print("Provider:", ctx["provider"])
    print("Document type:", ctx["document_type"])
    print("Source:", ctx["source_file"])
    print("Section:", ctx["section_path"])
    print(ctx["chunk_text"][:700])

Score: 0.5993006
Context ID: CTX-0208
Provider: neutral
Document type: architecture_pattern
Source: event_driven_document_ingestion_pattern.md
Section: Architecture Pattern: Event-Driven Document Ingestion > Generic components > Retrieval layer
### Retrieval layer

Stores searchable chunks and metadata.

Examples:
- Qdrant for local MVP.
- Azure AI Search for Azure architecture.
- Bedrock Knowledge Bases or OpenSearch for AWS architecture.
Score: 0.54532576
Context ID: CTX-0202
Provider: neutral
Document type: architecture_pattern
Source: event_driven_document_ingestion_pattern.md
Section: Architecture Pattern: Event-Driven Document Ingestion > Requirements
## Requirements

- Detect new uploaded documents.
- Trigger processing automatically.
- Extract or normalize document metadata.
- Extract text from documents.
- Split text into chunks.
- Index chunks into a retrieval system.
- Keep traceability to the original source document.
- Support retries or failure handling.
Score: 0.5349221


In [36]:
azure_contexts = retrieve_contexts(
    query="""
    Azure enterprise RAG architecture with document storage, ingestion,
    vector search, hybrid search, metadata filtering and agent deployment.
    """,
    provider="azure",
    top_k=5
)

for ctx in azure_contexts:
    print("=" * 120)
    print("Score:", ctx["score"])
    print("Context ID:", ctx["context_id"])
    print("Provider:", ctx["provider"])
    print("Document type:", ctx["document_type"])
    print("Source:", ctx["source_file"])
    print("Section:", ctx["section_path"])
    print(ctx["chunk_text"][:700])

Score: 0.7837925
Context ID: CTX-0126
Provider: azure
Document type: decision_record
Source: azure_ai_search_for_enterprise_rag.md
Section: Decision Record: Azure AI Search for Enterprise RAG > When to use this decision
## When to use this decision

Use Azure AI Search when:
- The organization already uses Azure.
- The solution needs managed retrieval.
- The system needs hybrid search.
- The system needs metadata filtering.
- The architecture should integrate with Azure Blob Storage.
- The architecture should integrate with Azure OpenAI.
- Operational overhead should be reduced.
Score: 0.76768315
Context ID: CTX-0116
Provider: azure
Document type: decision_record
Source: azure_ai_search_for_enterprise_rag.md
Section: Decision Record: Azure AI Search for Enterprise RAG > Requirements
## Requirements

- Store searchable document chunks.
- Support semantic retrieval.
- Support keyword retrieval.
- Support vector search.
- Support hybrid search.
- Support metadata filtering.
- Integrate wi

In [16]:
def format_context_block(contexts: list[dict]) -> str:
    """
    Format retrieved contexts into a grounded context block for agents.
    Each block keeps a context_id so the agent can cite its evidence.
    """
    blocks = []

    for ctx in contexts:
        block = f"""[{ctx['context_id']}]
Provider: {ctx['provider']}
Document type: {ctx['document_type']}
Source: {ctx['source_file']}
Section: {ctx['section_path']}

{ctx['chunk_text']}"""
        blocks.append(block)

    return "\n\n" + ("\n\n" + "-" * 100 + "\n\n").join(blocks)

In [17]:
print(format_context_block(azure_contexts[:2]))



[CTX-0126]
Provider: azure
Document type: decision_record
Source: azure_ai_search_for_enterprise_rag.md
Section: Decision Record: Azure AI Search for Enterprise RAG > When to use this decision

## When to use this decision

Use Azure AI Search when:
- The organization already uses Azure.
- The solution needs managed retrieval.
- The system needs hybrid search.
- The system needs metadata filtering.
- The architecture should integrate with Azure Blob Storage.
- The architecture should integrate with Azure OpenAI.
- Operational overhead should be reduced.

----------------------------------------------------------------------------------------------------

[CTX-0116]
Provider: azure
Document type: decision_record
Source: azure_ai_search_for_enterprise_rag.md
Section: Decision Record: Azure AI Search for Enterprise RAG > Requirements

## Requirements

- Store searchable document chunks.
- Support semantic retrieval.
- Support keyword retrieval.
- Support vector search.
- Support hybrid 

In [18]:
def build_context_pack(
    user_idea: str,
    top_k_provider: int = 5,
    top_k_neutral: int = 4
) -> dict:
    """
    Build separated context packs for Azure Agent and AWS Agent.
    
    Azure Agent receives:
    - Azure contexts
    - Neutral architecture pattern contexts
    
    AWS Agent receives:
    - AWS contexts
    - Neutral architecture pattern contexts
    """

    azure_query = f"""
    Azure architecture for this project:
    {user_idea}
    
    Focus on RAG, document ingestion, retrieval, managed search,
    metadata filtering, agents and deployment.
    """

    aws_query = f"""
    AWS architecture for this project:
    {user_idea}
    
    Focus on RAG, document ingestion, retrieval, S3, Bedrock,
    Knowledge Bases, agents and deployment.
    """

    neutral_query = f"""
    Architecture pattern for this project:
    {user_idea}
    
    Focus on document assistant, ingestion pipeline, retrieval layer,
    agent layer, local MVP and decision records.
    """

    azure_contexts = retrieve_contexts(
        query=azure_query,
        provider="azure",
        top_k=top_k_provider
    )

    aws_contexts = retrieve_contexts(
        query=aws_query,
        provider="aws",
        top_k=top_k_provider
    )

    neutral_contexts = retrieve_contexts(
        query=neutral_query,
        provider="neutral",
        top_k=top_k_neutral
    )

    return {
        "user_idea": user_idea,

        "azure_contexts": azure_contexts,
        "aws_contexts": aws_contexts,
        "neutral_contexts": neutral_contexts,

        "azure_context_block": format_context_block(
            azure_contexts + neutral_contexts
        ),

        "aws_context_block": format_context_block(
            aws_contexts + neutral_contexts
        ),
    }

In [19]:
user_idea = """
I want to build a system where users upload project documents.
The system processes the documents, indexes them, and then two agents propose
one architecture using Azure and another architecture using AWS.
A judge agent compares both proposals.
The MVP must run locally first and avoid paid cloud resources.
"""

context_pack = build_context_pack(user_idea)

print("AZURE CONTEXT BLOCK")
print("=" * 120)
print(context_pack["azure_context_block"])

print("\n\nAWS CONTEXT BLOCK")
print("=" * 120)
print(context_pack["aws_context_block"])

AZURE CONTEXT BLOCK


[CTX-0053]
Provider: azure
Document type: cloud_reference
Source: azure_agentic_app.md
Section: Azure Solution: Multi-Agent Architecture Advisor > Architecture

## Architecture

The recommended MVP architecture is:

1. A user submits a project idea through a simple frontend or notebook.
2. A FastAPI backend receives the request.
3. The backend executes a Microsoft Agent Framework workflow.
4. The RequirementsAgent extracts structured requirements.
5. The AzureSolutionAgent proposes an Azure architecture.
6. The AWSSolutionAgent proposes an AWS architecture.
7. The CloudComparisonAgent compares both options.
8. The FinalDecisionAgent recommends a final architecture.
9. The generated recommendation is stored as Markdown in Azure Blob Storage.
10. Metadata and architecture decisions are stored in Azure SQL Database.
11. Logs, errors and latency are monitored with Application Insights.

----------------------------------------------------------------------------------

In [20]:
def inspect_context_pack(context_pack: dict):
    print("Azure contexts")
    print("-" * 80)
    for ctx in context_pack["azure_contexts"]:
        print(
            ctx["context_id"],
            "|",
            ctx["provider"],
            "|",
            ctx["document_type"],
            "|",
            ctx["source_file"],
            "|",
            ctx["section_path"]
        )

    print("\nNeutral contexts")
    print("-" * 80)
    for ctx in context_pack["neutral_contexts"]:
        print(
            ctx["context_id"],
            "|",
            ctx["provider"],
            "|",
            ctx["document_type"],
            "|",
            ctx["source_file"],
            "|",
            ctx["section_path"]
        )

    print("\nAWS contexts")
    print("-" * 80)
    for ctx in context_pack["aws_contexts"]:
        print(
            ctx["context_id"],
            "|",
            ctx["provider"],
            "|",
            ctx["document_type"],
            "|",
            ctx["source_file"],
            "|",
            ctx["section_path"]
        )


inspect_context_pack(context_pack)

Azure contexts
--------------------------------------------------------------------------------
CTX-0053 | azure | cloud_reference | azure_agentic_app.md | Azure Solution: Multi-Agent Architecture Advisor > Architecture
CTX-0050 | azure | cloud_reference | azure_agentic_app.md | Azure Solution: Multi-Agent Architecture Advisor > Context
CTX-0055 | azure | cloud_reference | azure_agentic_app.md | Azure Solution: Multi-Agent Architecture Advisor > Scalable version
CTX-0054 | azure | cloud_reference | azure_agentic_app.md | Azure Solution: Multi-Agent Architecture Advisor > MVP recommendation
CTX-0057 | azure | cloud_reference | azure_agentic_app.md | Azure Solution: Multi-Agent Architecture Advisor > Cons

Neutral contexts
--------------------------------------------------------------------------------
CTX-0198 | neutral | architecture_pattern | architecture_patterns.md | Architecture Patterns for AI Agent Applications > Decision rule: Azure vs AWS
CTX-0208 | neutral | architecture_patte

In [21]:
def validate_context_pack(context_pack: dict) -> dict:
    """
    Validate that provider-specific context packs are clean.
    """
    azure_invalid = [
        ctx for ctx in context_pack["azure_contexts"]
        if ctx["provider"] != "azure"
    ]

    aws_invalid = [
        ctx for ctx in context_pack["aws_contexts"]
        if ctx["provider"] != "aws"
    ]

    neutral_invalid = [
        ctx for ctx in context_pack["neutral_contexts"]
        if ctx["provider"] != "neutral"
    ]

    return {
        "azure_contexts_valid": len(azure_invalid) == 0,
        "aws_contexts_valid": len(aws_invalid) == 0,
        "neutral_contexts_valid": len(neutral_invalid) == 0,
        "azure_invalid": azure_invalid,
        "aws_invalid": aws_invalid,
        "neutral_invalid": neutral_invalid,
    }


validation = validate_context_pack(context_pack)
validation

{'azure_contexts_valid': True,
 'aws_contexts_valid': True,
 'neutral_contexts_valid': True,
 'azure_invalid': [],
 'aws_invalid': [],
 'neutral_invalid': []}

In [37]:
import json

OUTPUT_DIR = BASE_DIR / "data" / "context_packs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

context_pack_output_path = OUTPUT_DIR / "sample_context_pack.json"

with open(context_pack_output_path, "w", encoding="utf-8") as f:
    json.dump(context_pack, f, indent=2, ensure_ascii=False)

print("Saved context pack to:", context_pack_output_path)

Saved context pack to: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\context_packs\sample_context_pack.json
